# Citation Sentiment Analysis => A0–A7 Factorial Experiments

**Fixed Base:** Athar + ACL-OCL + real Xu2015 Clinical.

The fixed Base participates in the same leakage-safe, group-aware **60% train / 20% validation / 20% test** split with `SEED = 42`.

- **A0/A1:** Base = Athar + ACL-OCL + Clinical
- **A2/A3:** Base + Journal real
- **A4/A5:** Base + Synthetic
- **A6/A7:** Base + Journal real + Synthetic

Journal-real and Synthetic data are added **only to train**. Clinical data are part of the Base before splitting.

In [ ]:
import os
import hashlib

# Set WANDB_API_KEY as an environment variable (never commit an API key).

# Ορισμός TensorBoard Log Directory μέσω Environment Variable
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""Εγκατάσταση απαραίτων στοιχείων."""

!pip install accelerate -U
!pip install transformers[torch]
!pip install bitsandbytes

## 2. Imports, NLTK resources and reproducibility

In [ ]:

"""Συμπερίληψη απαραίτων βιβλιοθηκών και Python modules."""

import pandas as pd
import math
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DebertaV2Tokenizer, DebertaForSequenceClassification, DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification, XLNetTokenizer, XLNetForSequenceClassification
from transformers import ElectraTokenizer, ElectraForSequenceClassification, AlbertTokenizer, AlbertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import time
import random
import re
import bitsandbytes as bnb
from torch.optim import Adam
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import string
from nltk.corpus import wordnet

# Κατέβασμα των απαραίτητων δεδομένων από το nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # Για σωστό lemmatization
nltk.download('averaged_perceptron_tagger')  # POS tagging
nltk.download('punkt')  # Tokenizer
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Για αναπαραγωγιμότητα. Αν το seed είναι το ίδιο, παίρνεις την ίδια ακολουθία τυχαίων επιλογών. Θέλω να παίρνω κάθε φορά το ίδιο test set.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

"""Οι συναρτήσεις καθαρισμού σε αυτό το κελί είναι προαιρετικές. Καλύτερα να μην εφαρμόζονται σε LLMs."""

## 3. Optional text-cleaning helpers

In [ ]:

# Αφαίρεση stopwords, σημείων στίξης και lemmatization.
def get_wordnet_pos(word):
    """ Μετατρέπει τις POS tags από το nltk στη μορφή που απαιτεί το WordNet Lemmatizer """
    tag = pos_tag([word])[0][1][0].upper()  # Παίρνουμε το πρώτο γράμμα του POS tag
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}  # Μετατροπή σε WordNet μορφή
    return tag_dict.get(tag, wordnet.NOUN)  # Αν δεν υπάρχει στο dict, υποθέτουμε NOUN (ουσιαστικό)

# Έκδοση της clean_text() με Lemmatization + lowercase
def clean_text(text, language="english"):
    """
    Καθαρίζει ένα κείμενο από stopwords, σημεία στίξης και εκτελεί σωστό lemmatization.
    Οι λέξεις μετατρέπονται σε πεζά γράμματα.
    """
    stop_words = set(stopwords.words(language))
    lemmatizer = WordNetLemmatizer()

    if isinstance(text, str):
        # Μετατροπή όλων των χαρακτήρων σε πεζά
        text = text.lower()

        # Αφαίρεση σημείων στίξης
        text = text.translate(str.maketrans('', '', string.punctuation))

        # Tokenization (Διαχωρισμός σε λέξεις)
        words = word_tokenize(text)

        # Αφαίρεση stopwords
        filtered_words = [word for word in words if word not in stop_words]

        # Lemmatization με βάση το σωστό POS tag
        lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_words]

        return " ".join(lemmatized_words)

    return text  # Αν δεν είναι string, επιστρέφει το αρχικό δεδομένο.

# Συνάρτηση που εφαρμόζει την clean_text στο dataset.
def remove_stopwords_from_csv(df, text_column="sentence", language="english"):
    """
    Εφαρμόζει την clean_text σε μια στήλη του DataFrame για να αφαιρέσει stopwords, σημεία στίξης και να εκτελέσει lemmatization.

    Args:
        df (pd.DataFrame): Το DataFrame που περιέχει τα δεδομένα.
        text_column (str): Το όνομα της στήλης που περιέχει το κείμενο.
        language (str): Η γλώσσα των stopwords (default: 'english').

    Returns:
        pd.DataFrame: Το καθαρισμένο DataFrame.
    """
    df[text_column] = df[text_column].apply(lambda x: clean_text(x, language))
    return df

"""Στα LLMs καλύτερα εφαρμογή μόνο lowercasing."""

def clean_text_only(text, language="english"):
    text = text.lower()
    return text

def lower_words_only(df, text_column="sentence", language="english"):
    df[text_column] = df[text_column].apply(lambda x: clean_text_only(x, language))
    return df

"""Φόρτωση και ανάλυση των δεδομένων."""


## 4. Experimental design and dataset configuration

In [ ]:
# ============================================================
# FACTORIAL EXPERIMENT DESIGN: 4 DATA CONDITIONS x 2 LOSS STRATEGIES
# ============================================================
#
# FIXED BASE CORPUS:
#   Athar + ACL-OCL + real Xu2015 Clinical citations.
#
# IMPORTANT: the ENTIRE fixed base corpus participates in the same
# train/validation/test split. Therefore Clinical rows are allowed to
# appear in train, validation, or test, subject to Source_Paper_ID grouping.
#
# DATASET_CHOICE controls ONLY which EXTRA data are added to TRAIN:
#   1 = BASE
#       Athar + ACL-OCL + Clinical only.
#   2 = JOURNAL_AUGMENTED
#       Base train + real journal citations.
#   3 = SYNTHETIC_AUGMENTED
#       Base train + synthetic citations.
#   4 = COMBINED_AUGMENTED
#       Base train + Journal real + Synthetic.
#
# LOSS_CHOICE controls ONLY the loss function:
#   1 = STANDARD_CE
#   2 = WEIGHTED_CE
#
# Fixed A0-A7 design:
#   A0 = Base (Athar + ACL-OCL + Clinical)             + Standard CE
#   A1 = Base (Athar + ACL-OCL + Clinical)             + Weighted CE
#   A2 = Base + Journal real                            + Standard CE
#   A3 = Base + Journal real                            + Weighted CE
#   A4 = Base + Synthetic                               + Standard CE
#   A5 = Base + Synthetic                               + Weighted CE
#   A6 = Base + Journal real + Synthetic                + Standard CE
#   A7 = Base + Journal real + Synthetic                + Weighted CE
#
# IMPORTANT EXPERIMENTAL CONTROLS:
#   - The train/validation/test split is ALWAYS created from the same fixed
#     Base + Clinical corpus.
#   - The split is approximately 60% train / 20% validation / 20% test.
#   - The split is group-aware with Source_Paper_ID and SEED=42.
#   - Clinical rows are part of the base BEFORE splitting and may therefore
#     occur in train, validation, or test.
#   - Journal real and Synthetic data are added ONLY to train.
#   - Validation and test therefore remain identical across A0-A7.
#   - *learning_rate is fixed at 2e-5 for all A1-A7 runs after A0 tuning.
#   - Weighted CE weights, when used, are recomputed from the FINAL train_data
#     of that specific experiment and NEVER from validation/test.
# ============================================================

# This file already contains Athar + ACL-OCL + real Xu2015 Clinical rows.
BASE_CLINICAL_DATASET_PATH = (
    "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/"
    "athar_acl_ocl_outputs_with_clinical/_final_citation_non_citation_plus_clinical_real.csv"
)

JOURNAL_DATASET_PATH = (
    "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/"
    "journal_citation_sentiment_real.csv"
)

SYNTHETIC_DATASET_PATH = (
    "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/"
    "synthetic_citation_sentiment_6995.csv"
)

print("\nΕπίλεξε TRAINING DATA condition:")
print("1 - BASE: Athar + ACL-OCL + Clinical (fixed base)")
print("2 - JOURNAL_AUGMENTED: + journal real μόνο στο train")
print("3 - SYNTHETIC_AUGMENTED: + synthetic μόνο στο train")
print("4 - COMBINED_AUGMENTED: + Journal real + Synthetic μόνο στο train")

DATASET_CHOICE = input("Dataset επιλογή (1, 2, 3 ή 4): ").strip()

if DATASET_CHOICE not in {"1", "2", "3", "4"}:
    raise ValueError("Μη έγκυρη επιλογή dataset. Δώσε 1, 2, 3 ή 4.")

print("\nΕπίλεξε LOSS strategy:")
print("1 - STANDARD_CE: κανονικό Cross Entropy")
print("2 - WEIGHTED_CE: class-weighted Cross Entropy")

LOSS_CHOICE = input("Loss επιλογή (1 ή 2): ").strip()

if LOSS_CHOICE not in {"1", "2"}:
    raise ValueError("Μη έγκυρη επιλογή loss. Δώσε 1 ή 2.")

# Mapping των δύο ανεξάρτητων επιλογών στο προκαθορισμένο A0-A7 experiment.
EXPERIMENT_MAP = {
    ("1", "1"): ("A0", "BASE_CLINICAL_STANDARD_CE"),
    ("1", "2"): ("A1", "BASE_CLINICAL_WEIGHTED_CE"),
    ("2", "1"): ("A2", "JOURNAL_AUGMENTED_STANDARD_CE"),
    ("2", "2"): ("A3", "JOURNAL_AUGMENTED_WEIGHTED_CE"),
    ("3", "1"): ("A4", "SYNTHETIC_AUGMENTED_STANDARD_CE"),
    ("3", "2"): ("A5", "SYNTHETIC_AUGMENTED_WEIGHTED_CE"),
    ("4", "1"): ("A6", "COMBINED_AUGMENTED_STANDARD_CE"),
    ("4", "2"): ("A7", "COMBINED_AUGMENTED_WEIGHTED_CE"),
}

EXPERIMENT_ID, CONDITION_TAG = EXPERIMENT_MAP[(DATASET_CHOICE, LOSS_CHOICE)]
EXPERIMENT_TAG = f"{EXPERIMENT_ID}_{CONDITION_TAG}"
USE_WEIGHTED_CE = LOSS_CHOICE == "2"

DATASET_LABEL = {
    "1": "BASE_CLINICAL",
    "2": "JOURNAL_AUGMENTED",
    "3": "SYNTHETIC_AUGMENTED",
    "4": "COMBINED_AUGMENTED",
}[DATASET_CHOICE]

LOSS_LABEL = "WEIGHTED_CE" if USE_WEIGHTED_CE else "STANDARD_CE"

print("\n" + "=" * 70)
print("SELECTED EXPERIMENT")
print("=" * 70)
print("Experiment ID:", EXPERIMENT_ID)
print("Training data condition:", DATASET_LABEL)
print("Loss strategy:", LOSS_LABEL)
print("Experiment tag:", EXPERIMENT_TAG)
print("=" * 70 + "\n")

REQUIRED_COLUMNS = [
    "Source_Paper_ID",
    "sentence",
    "is_citation",
    "polarity",
    "source",
    "sampling_status",
]


## 5. Helper for Journal/Synthetic external citation datasets

In [ ]:
# ------------------------------------------------------------
# Helper για external citation datasets που μπορεί να έχουν είτε
# πλήρες schema είτε μόνο sentence / is_citation / polarity.
# ------------------------------------------------------------
def prepare_external_citation_rows(
    raw_df,
    dataset_name,
    source_name,
    sampling_status,
    base_sentence_set,
    allowed_labels={0, 2},
):
    required_core = {"sentence", "is_citation", "polarity"}
    missing_core = required_core - set(raw_df.columns)

    if missing_core:
        raise ValueError(
            f"Λείπουν στήλες από το {dataset_name}: {sorted(missing_core)}"
        )

    rows = raw_df.copy()
    rows["sentence"] = rows["sentence"].astype(str).str.strip()

    # Μετατροπή σε αριθμητικές τιμές με σαφή αποτυχία για μη έγκυρες τιμές.
    rows["is_citation"] = pd.to_numeric(
        rows["is_citation"], errors="raise"
    ).astype(int)
    rows["polarity"] = pd.to_numeric(
        rows["polarity"], errors="raise"
    ).astype(int)

    # Το external corpus που προστίθεται εδώ πρέπει να αποτελείται από citations.
    if not (rows["is_citation"] == 1).all():
        invalid_count = int((rows["is_citation"] != 1).sum())
        raise ValueError(
            f"Το {dataset_name} περιέχει {invalid_count} rows με is_citation != 1."
        )

    # Τα augmentation corpora του συγκεκριμένου experiment περιέχουν
    # μόνο Negative (0) και Positive (2), όχι Neutral (1).
    invalid_labels = set(rows["polarity"].unique()) - set(allowed_labels)
    if invalid_labels:
        raise ValueError(
            f"Το {dataset_name} περιέχει μη έγκυρα polarity labels: "
            f"{sorted(invalid_labels)}. Επιτρέπονται μόνο {sorted(allowed_labels)}."
        )

    # Αφαίρεση κενών sentences.
    rows = rows[rows["sentence"].ne("")].copy()

    # Αν η ίδια sentence έχει διαφορετικό polarity μέσα στο ίδιο corpus,
    # αφαιρούνται όλες οι εκδοχές της για να μην εισαχθεί label conflict.
    label_counts = rows.groupby("sentence")["polarity"].nunique()
    conflicting_sentences = set(label_counts[label_counts > 1].index)

    if conflicting_sentences:
        print(
            f"{dataset_name} conflicting sentences που αφαιρούνται:",
            len(conflicting_sentences),
        )
        rows = rows[
            ~rows["sentence"].isin(conflicting_sentences)
        ].copy()

    before_dedup = len(rows)
    rows = (
        rows
        .drop_duplicates(subset="sentence", keep="first")
        .reset_index(drop=True)
    )

    print(
        f"{dataset_name} duplicate rows που αφαιρέθηκαν:",
        before_dedup - len(rows),
    )

    # Hard overlap protection με ολόκληρο το base corpus.
    overlap_mask = rows["sentence"].astype(str).isin(base_sentence_set)

    if overlap_mask.any():
        print(
            "ΠΡΟΣΟΧΗ: αφαιρούνται",
            int(overlap_mask.sum()),
            f"rows από {dataset_name} που υπάρχουν ήδη στο base dataset.",
        )
        rows = rows.loc[~overlap_mask].reset_index(drop=True)

    # Ενοποιημένο metadata schema.
    # Για journal/synthetic external datasets χρησιμοποιούμε σταθερό source label,
    # ώστε οι leakage checks και τα counts να είναι απολύτως σαφή.
    if "Source_Paper_ID" not in rows.columns:
        rows["Source_Paper_ID"] = f"{source_name}_SOURCE_ID_UNAVAILABLE"
    else:
        rows["Source_Paper_ID"] = rows["Source_Paper_ID"].fillna(
            f"{source_name}_SOURCE_ID_UNAVAILABLE"
        )

    rows["source"] = source_name
    rows["sampling_status"] = sampling_status

    return rows[REQUIRED_COLUMNS].copy()


## 6. Load fixed Base = Athar + ACL-OCL + Clinical

In [ ]:
# ------------------------------------------------------------
# FIXED BASE = ATHAR + ACL-OCL + CLINICAL
#
# Το Base + Clinical dataset φορτώνεται ΠΑΝΤΑ και αποτελεί τη ΜΟΝΑΔΙΚΗ
# βάση για το train/validation/test split. Το Clinical ΔΕΝ είναι πλέον
# external augmentation: συμμετέχει κανονικά στο group-aware split.
# ------------------------------------------------------------
base_df = pd.read_csv(BASE_CLINICAL_DATASET_PATH)

missing_cols = set(REQUIRED_COLUMNS) - set(base_df.columns)
if missing_cols:
    raise ValueError(
        f"Λείπουν στήλες από το Base + Clinical dataset: {sorted(missing_cols)}"
    )

base_df = base_df[REQUIRED_COLUMNS].copy()
base_df["sentence"] = base_df["sentence"].astype(str).str.strip()
base_df["is_citation"] = pd.to_numeric(
    base_df["is_citation"], errors="raise"
).astype(int)
base_df["polarity"] = pd.to_numeric(
    base_df["polarity"], errors="raise"
).astype(int)

# Αφαίρεση κενών sentences πριν από το deduplication.
base_df = base_df[base_df["sentence"].ne("")].copy()

print("\nFIXED BASE DATASET = ATHAR + ACL-OCL + CLINICAL")
print("Αρχικές γραμμές:", len(base_df))
print("\nRows ανά source πριν το deduplication:")
print(base_df["source"].value_counts(dropna=False))

clinical_source_name = "Clinical_Citation_Sentiment_Xu2015"
clinical_total_before_dedup = int(
    base_df["source"].eq(clinical_source_name).sum()
)
if clinical_total_before_dedup == 0:
    raise ValueError(
        "Δεν βρέθηκαν Clinical_Citation_Sentiment_Xu2015 rows στο fixed base dataset."
    )

base_duplicates = base_df["sentence"].duplicated().sum()
print("Διπλότυπες sentence:", base_duplicates)

# Ίδια sentence-level deduplication λογική με τον αρχικό κώδικα.
base_df = (
    base_df
    .drop_duplicates(subset="sentence", keep="first")
    .reset_index(drop=True)
)

print("Base + Clinical rows μετά το deduplication:", len(base_df))
print("\nRows ανά source μετά το deduplication:")
print(base_df["source"].value_counts(dropna=False))
print("\nBase + Clinical polarity distribution:")
print(base_df["polarity"].value_counts().sort_index())

clinical_total = int(base_df["source"].eq(clinical_source_name).sum())
print("\nClinical rows μέσα στο fixed base μετά το deduplication:", clinical_total)
print(
    base_df.loc[base_df["source"].eq(clinical_source_name), "polarity"]
    .value_counts()
    .sort_index()
)

base_sentence_set = set(base_df["sentence"].astype(str))


## 7. Load Journal/Synthetic data and descriptive statistics

In [ ]:
# ------------------------------------------------------------
# JOURNAL REAL DATA
# Φορτώνονται στις επιλογές 2 και 4.
# Προστίθενται ΜΟΝΟ στο train μετά το fixed base split.
# ------------------------------------------------------------
journal_rows = pd.DataFrame(columns=REQUIRED_COLUMNS)

if DATASET_CHOICE in {"2", "4"}:
    journal_raw = pd.read_csv(JOURNAL_DATASET_PATH)

    journal_rows = prepare_external_citation_rows(
        raw_df=journal_raw,
        dataset_name="JOURNAL REAL",
        source_name="Journal_Citation_Sentiment_Real",
        sampling_status="external_real_journal",
        base_sentence_set=base_sentence_set,
    )

    print("\nJOURNAL REAL ROWS ΠΟΥ ΕΙΝΑΙ ΔΙΑΘΕΣΙΜΕΣ ΓΙΑ ΤΟ TRAIN")
    print("Rows:", len(journal_rows))
    print(journal_rows["polarity"].value_counts().sort_index())

# ------------------------------------------------------------
# SYNTHETIC DATA
# Φορτώνονται στις επιλογές 3 και 4.
# Προστίθενται ΜΟΝΟ στο train μετά το fixed base split.
# ------------------------------------------------------------
synthetic_rows = pd.DataFrame(columns=REQUIRED_COLUMNS)

if DATASET_CHOICE in {"3", "4"}:
    synthetic_raw = pd.read_csv(SYNTHETIC_DATASET_PATH)

    synthetic_rows = prepare_external_citation_rows(
        raw_df=synthetic_raw,
        dataset_name="SYNTHETIC",
        source_name="Synthetic_Citation_Sentiment",
        sampling_status="external_synthetic",
        base_sentence_set=base_sentence_set,
    )

    print("\nSYNTHETIC ROWS ΠΟΥ ΕΙΝΑΙ ΔΙΑΘΕΣΙΜΕΣ ΓΙΑ ΤΟ TRAIN")
    print("Rows:", len(synthetic_rows))
    print(synthetic_rows["polarity"].value_counts().sort_index())

# ------------------------------------------------------------
# COMBINED EXTERNAL DATA = JOURNAL REAL + SYNTHETIC
# Χρησιμοποιούνται μόνο στην επιλογή 4.
#
# Clinical ΔΕΝ βρίσκεται εδώ: είναι ήδη μέρος του fixed base και έχει split.
#
# Κανόνες για overlap μεταξύ Journal και Synthetic:
# 1. Αν ίδια sentence έχει διαφορετικό polarity -> αφαιρούνται όλες οι εκδοχές.
# 2. Αν ίδια sentence έχει ίδιο polarity και στα δύο -> κρατάμε Journal real
#    κατά προτεραιότητα έναντι Synthetic.
# ------------------------------------------------------------
combined_external_rows = pd.DataFrame(columns=REQUIRED_COLUMNS)

if DATASET_CHOICE == "4":
    external_pool = pd.concat(
        [journal_rows, synthetic_rows],
        ignore_index=True,
    )

    cross_source_label_counts = (
        external_pool
        .groupby("sentence")["polarity"]
        .nunique()
    )

    cross_source_conflicts = set(
        cross_source_label_counts[
            cross_source_label_counts > 1
        ].index
    )

    if cross_source_conflicts:
        print(
            "\nCross-source conflicting sentences που αφαιρούνται:",
            len(cross_source_conflicts),
        )
        external_pool = external_pool[
            ~external_pool["sentence"].isin(cross_source_conflicts)
        ].copy()

    source_priority = {
        "Journal_Citation_Sentiment_Real": 0,
        "Synthetic_Citation_Sentiment": 1,
    }

    external_pool["_source_priority"] = (
        external_pool["source"]
        .map(source_priority)
        .fillna(99)
        .astype(int)
    )

    combined_before_cross_dedup = len(external_pool)

    combined_external_rows = (
        external_pool
        .sort_values(["sentence", "_source_priority"])
        .drop_duplicates(subset="sentence", keep="first")
        .drop(columns="_source_priority")
        .reset_index(drop=True)
    )

    print(
        "Cross-source same-sentence duplicates που αφαιρέθηκαν:",
        combined_before_cross_dedup - len(combined_external_rows),
    )

    print("\nCOMBINED JOURNAL + SYNTHETIC ROWS ΠΟΥ ΘΑ ΠΡΟΣΤΕΘΟΥΝ ΣΤΟ TRAIN")
    print("Total rows:", len(combined_external_rows))
    print("\nBy source:")
    print(combined_external_rows["source"].value_counts())
    print("\nBy polarity:")
    print(combined_external_rows["polarity"].value_counts().sort_index())

# ------------------------------------------------------------
# Το df χρησιμοποιείται μόνο για descriptive statistics/plot.
# ΔΕΝ χρησιμοποιείται για το split.
# ------------------------------------------------------------
if DATASET_CHOICE == "1":
    df = base_df.copy()

elif DATASET_CHOICE == "2":
    df = pd.concat(
        [base_df, journal_rows],
        ignore_index=True,
    )

elif DATASET_CHOICE == "3":
    df = pd.concat(
        [base_df, synthetic_rows],
        ignore_index=True,
    )

elif DATASET_CHOICE == "4":
    df = pd.concat(
        [base_df, combined_external_rows],
        ignore_index=True,
    )

print("\nSelected experiment:", EXPERIMENT_TAG)
print("Rows για descriptive statistics:", len(df))

polarity_stats = (
    df["polarity"]
    .value_counts()
    .sort_index()
)

print("\nPolarity Statistics:")
print(polarity_stats)


from matplotlib.patches import Patch

# Ρυθμίσεις εμφάνισης
sns.set(style="whitegrid", context="paper", font_scale=1.0)

# Ορισμός χαρτογράφησης
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

plt.rcParams.update({'font.size': 11})

# Δημιουργία ραβδογράμματος
plt.figure(figsize=(6, 4))

ax = sns.barplot(
    x=polarity_stats.index,
    y=polarity_stats.values,
    hue=polarity_stats.index,
    legend=False,
    palette=["#89CFF0", "#FA8072", "#90EE90"]
)

# Προσαρμογή ετικετών στον άξονα x
ax.set_xticks(range(len(polarity_stats.index)))
ax.set_xticklabels([label_map[i] for i in polarity_stats.index], fontsize=11)

# Προσθήκη τίτλων και ετικετών
plt.xlabel('')
plt.ylabel('Number of sentences')

# Προσθήκη αριθμητικών τιμών πάνω στις μπάρες
for p in ax.patches:
    height = p.get_height()
    ax.text(
        p.get_x() + p.get_width() / 2.,
        height + max(polarity_stats.values) * 0.00,
        f'{int(height)}',
        ha='center', va='bottom', fontsize=10
    )

# Προσθήκη custom legend
colors = ["#89CFF0", "#FA8072", "#90EE90"]
labels = [label_map[i] for i in polarity_stats.index]
handles = [Patch(color=c, label=l) for c, l in zip(colors, labels)]
plt.legend(
    handles=handles,
    title='Sentiment',
    loc='upper right',
    fontsize=11,
    title_fontsize=11
)

# Ρύθμιση πλέγματος και περιθωρίων
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Αποθήκευση ως PDF υψηλής ανάλυσης, διαφορετικό αρχείο ανά experiment.
plt.savefig(
    f'/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/fig/polarity_distribution_{EXPERIMENT_TAG}.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()


## 8. Leakage-safe fixed Base split — 60/20/20

- **Athar / ACL-OCL:** grouped by the real `Source_Paper_ID`.
- **Clinical:** because the source file uses an unavailable-ID placeholder, the split groups Clinical target-swapped variants by a normalized citation context so near-duplicate variants remain in one split.
- `StratifiedGroupKFold`, `n_splits=5`, `SEED=42`.

In [ ]:
# ============================================================
# FIXED LEAKAGE-SAFE BASE SPLIT: 60% / 20% / 20%
# ============================================================
#
# Το split γίνεται ΠΑΝΤΑ και ΜΟΝΟ στο fixed base_df:
#   Athar + ACL-OCL + Clinical.
#
# Στόχος:
#   - περίπου 60% train / 20% validation / 20% test
#   - stratification ως προς polarity
#   - SEED = 42
#
# GROUPING:
#   Athar / ACL-OCL:
#       grouping με το πραγματικό Source_Paper_ID, όπως στον αρχικό κώδικα V1.
#       Άρα, όλες οι προτάσεις του ίδιου citing paper μένουν στο ίδιο split.
#
#   Clinical Xu2015:
#       το CSV δεν διαθέτει πραγματικό Source_Paper_ID: όλες οι Clinical rows
#       έχουν το placeholder CLINICAL_XU2015_SOURCE_ID_UNAVAILABLE.
#       Δεν μπορούμε λοιπόν να χρησιμοποιήσουμε αυτό το placeholder ως group,
#       γιατί τότε όλες οι Clinical rows θα κατέληγαν στο ίδιο split.
#
#       Ο έλεγχος του corpus έδειξε ότι δεν υπάρχουν exact/normalized duplicate
#       Clinical sentences, αλλά υπάρχουν target-swapped instances: το ίδιο
#       citation context εμφανίζεται με το [TARGET_CITATION] σε διαφορετική θέση.
#       Για να μην υπάρξει near-duplicate leakage, δημιουργούμε Clinical group
#       από το citation context αφού αντικαταστήσουμε TARGET/OTHER με [CITATION].
#       Έτσι target-swapped variants μένουν μαζί, ενώ ανεξάρτητα Clinical
#       contexts μπορούν να μοιραστούν κανονικά στα διαφορετικά splits.
#
# Journal real / Synthetic ΔΕΝ συμμετέχουν στο split και προστίθενται
# αργότερα ΜΟΝΟ στο train ανάλογα με το DATASET_CHOICE.
#
# Με ίδιο Base + Clinical CSV, SEED=42 και ίδιες παραμέτρους, validation/test
# παραμένουν ακριβώς ίδια σε όλα τα A0-A7 experiments.
# ============================================================

if "Source_Paper_ID" not in base_df.columns:
    raise ValueError(
        "Η στήλη Source_Paper_ID είναι απαραίτητη για το Athar/ACL-OCL group-aware split."
    )

# Για Athar / ACL-OCL απαιτείται έγκυρο πραγματικό Source_Paper_ID.
nonclinical_mask = ~base_df["source"].eq(clinical_source_name)

if base_df.loc[nonclinical_mask, "Source_Paper_ID"].isna().any():
    raise ValueError(
        "Βρέθηκαν κενές Source_Paper_ID σε Athar/ACL-OCL. "
        "Δεν μπορεί να γίνει ασφαλές group-aware split."
    )

if (
    base_df.loc[nonclinical_mask, "Source_Paper_ID"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise ValueError(
        "Βρέθηκαν κενά Source_Paper_ID strings σε Athar/ACL-OCL. "
        "Δεν μπορεί να γίνει ασφαλές group-aware split."
    )

# ------------------------------------------------------------
# Leakage-safe split group
# ------------------------------------------------------------
# Athar / ACL-OCL: κρατάμε ΑΚΡΙΒΩΣ τη λογική Source_Paper_ID.
base_df["_split_group"] = base_df["Source_Paper_ID"].astype(str)

clinical_mask = base_df["source"].eq(clinical_source_name)

def clinical_citation_context_key(text):
    """
    Δημιουργεί leakage-safe group key για Clinical citation instances.

    TARGET και OTHER αντικαθίστανται από κοινό marker ώστε target-swapped
    versions του ίδιου citation context να πάρουν το ίδιο split group.
    """
    text = str(text)
    text = text.replace("[TARGET_CITATION]", "[CITATION]")
    text = text.replace("[OTHER_CITATION]", "[CITATION]")
    text = re.sub(r"\\s+", " ", text).strip().lower()
    return text

clinical_context = (
    base_df.loc[clinical_mask, "sentence"]
    .map(clinical_citation_context_key)
)

base_df.loc[clinical_mask, "_split_group"] = clinical_context.map(
    lambda x: "CLINICAL_CONTEXT_" + hashlib.sha256(
        x.encode("utf-8")
    ).hexdigest()
)

clinical_group_sizes = (
    base_df.loc[clinical_mask]
    .groupby("_split_group")
    .size()
)

print("\nCLINICAL LEAKAGE-SAFE GROUP CHECK")
print("Clinical rows:", int(clinical_mask.sum()))
print("Clinical unique context groups:", int(clinical_group_sizes.size))
print(
    "Clinical rows σε multi-instance target-aware groups:",
    int(clinical_group_sizes[clinical_group_sizes > 1].sum()),
)
print(
    "Clinical multi-instance groups:",
    int((clinical_group_sizes > 1).sum()),
)

# 5 folds => περίπου 20% ανά fold:
#   fold 0       -> TEST       (~20%)
#   fold 1       -> VALIDATION (~20%)
#   folds 2,3,4  -> TRAIN      (~60%)
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

fold_assignment = np.full(len(base_df), -1, dtype=int)

for fold_id, (_, fold_indices) in enumerate(
    sgkf.split(
        X=base_df,
        y=base_df["polarity"],
        groups=base_df["_split_group"],
    )
):
    fold_assignment[fold_indices] = fold_id

if (fold_assignment < 0).any():
    raise RuntimeError("Υπάρχουν base rows που δεν ανατέθηκαν σε fold.")

base_train_data = (
    base_df.loc[np.isin(fold_assignment, [2, 3, 4])]
    .copy()
    .reset_index(drop=True)
)

val_data = (
    base_df.loc[fold_assignment == 1]
    .copy()
    .reset_index(drop=True)
)

test_data = (
    base_df.loc[fold_assignment == 0]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# HARD GROUP-LEAKAGE CHECKS ΓΙΑ BASE + CLINICAL
# ------------------------------------------------------------
base_train_groups = set(base_train_data["_split_group"].astype(str))
val_groups = set(val_data["_split_group"].astype(str))
test_groups = set(test_data["_split_group"].astype(str))

assert base_train_groups.isdisjoint(val_groups), (
    "GROUP LEAKAGE: _split_group υπάρχει και σε base train και validation."
)
assert base_train_groups.isdisjoint(test_groups), (
    "GROUP LEAKAGE: _split_group υπάρχει και σε base train και test."
)
assert val_groups.isdisjoint(test_groups), (
    "GROUP LEAKAGE: _split_group υπάρχει και σε validation και test."
)

# Επιπλέον explicit check μόνο για Athar/ACL-OCL Source_Paper_ID.
athar_acl_train_ids = set(
    base_train_data.loc[
        ~base_train_data["source"].eq(clinical_source_name),
        "Source_Paper_ID",
    ].astype(str)
)
athar_acl_val_ids = set(
    val_data.loc[
        ~val_data["source"].eq(clinical_source_name),
        "Source_Paper_ID",
    ].astype(str)
)
athar_acl_test_ids = set(
    test_data.loc[
        ~test_data["source"].eq(clinical_source_name),
        "Source_Paper_ID",
    ].astype(str)
)

assert athar_acl_train_ids.isdisjoint(athar_acl_val_ids)
assert athar_acl_train_ids.isdisjoint(athar_acl_test_ids)
assert athar_acl_val_ids.isdisjoint(athar_acl_test_ids)

print("\n" + "=" * 70)
print("BASE + CLINICAL LEAKAGE-SAFE 60/20/20 SPLIT CHECK")
print("=" * 70)

print(
    "Athar/ACL-OCL unique Source_Paper_ID:",
    base_df.loc[nonclinical_mask, "Source_Paper_ID"].nunique(),
)
print("Clinical unique leakage-safe context groups:", clinical_group_sizes.size)

print("\nBase + Clinical rows ανά split:")
print(
    f"Train: {len(base_train_data)} "
    f"({len(base_train_data) / len(base_df):.2%})"
)
print(
    f"Validation: {len(val_data)} "
    f"({len(val_data) / len(base_df):.2%})"
)
print(
    f"Test: {len(test_data)} "
    f"({len(test_data) / len(base_df):.2%})"
)

print("\nBase + Clinical polarity distribution - TRAIN:")
print(base_train_data["polarity"].value_counts().sort_index())
print(base_train_data["polarity"].value_counts(normalize=True).sort_index())

print("\nBase + Clinical polarity distribution - VALIDATION:")
print(val_data["polarity"].value_counts().sort_index())
print(val_data["polarity"].value_counts(normalize=True).sort_index())

print("\nBase + Clinical polarity distribution - TEST:")
print(test_data["polarity"].value_counts().sort_index())
print(test_data["polarity"].value_counts(normalize=True).sort_index())

# Clinical rows are EXPECTED to be allowed in all three splits.
print("\nClinical rows ανά split:")
print(
    "Train:",
    int(base_train_data["source"].eq(clinical_source_name).sum())
)
print(
    "Validation:",
    int(val_data["source"].eq(clinical_source_name).sum())
)
print(
    "Test:",
    int(test_data["source"].eq(clinical_source_name).sum())
)

clinical_split_total = (
    int(base_train_data["source"].eq(clinical_source_name).sum())
    + int(val_data["source"].eq(clinical_source_name).sum())
    + int(test_data["source"].eq(clinical_source_name).sum())
)
assert clinical_split_total == clinical_total, (
    "Clinical accounting error: τα clinical rows των splits δεν ισούνται "
    "με τα clinical rows του fixed base."
)

print("\nLeakage-safe split-group overlap checks:")
print("Base train ∩ Validation:", len(base_train_groups & val_groups))
print("Base train ∩ Test:", len(base_train_groups & test_groups))
print("Validation ∩ Test:", len(val_groups & test_groups))

print("\nAthar/ACL-OCL Source_Paper_ID overlap checks:")
print("Train ∩ Validation:", len(athar_acl_train_ids & athar_acl_val_ids))
print("Train ∩ Test:", len(athar_acl_train_ids & athar_acl_test_ids))
print("Validation ∩ Test:", len(athar_acl_val_ids & athar_acl_test_ids))
print("=" * 70 + "\n")


## 9. Build final training set, leakage checks and split signatures

In [ ]:
# ------------------------------------------------------------
# FINAL TRAIN DATA: add only Journal/Synthetic according to condition.
# ------------------------------------------------------------
if DATASET_CHOICE == "1":
    train_data = (
        base_train_data
        .copy()
        .reset_index(drop=True)
    )

elif DATASET_CHOICE == "2":
    # Journal real μπαίνει ΜΟΝΟ στο train.
    train_data = pd.concat(
        [base_train_data, journal_rows],
        ignore_index=True,
    )
    train_data = (
        train_data
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

elif DATASET_CHOICE == "3":
    # Synthetic μπαίνει ΜΟΝΟ στο train.
    train_data = pd.concat(
        [base_train_data, synthetic_rows],
        ignore_index=True,
    )
    train_data = (
        train_data
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

elif DATASET_CHOICE == "4":
    # Journal real + Synthetic μπαίνουν ΜΟΝΟ στο train.
    train_data = pd.concat(
        [base_train_data, combined_external_rows],
        ignore_index=True,
    )
    train_data = (
        train_data
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

# ------------------------------------------------------------
# HARD LEAKAGE CHECKS FOR TRAIN-ONLY EXTERNAL DATA
# Clinical is intentionally NOT checked here because Clinical is part of base.
# ------------------------------------------------------------
if DATASET_CHOICE in {"2", "4"}:
    assert not val_data["source"].eq("Journal_Citation_Sentiment_Real").any()
    assert not test_data["source"].eq("Journal_Citation_Sentiment_Real").any()

if DATASET_CHOICE in {"3", "4"}:
    assert not val_data["source"].eq("Synthetic_Citation_Sentiment").any()
    assert not test_data["source"].eq("Synthetic_Citation_Sentiment").any()

# Sentence-level leakage checks for train-only external corpora.
if DATASET_CHOICE in {"2", "4"}:
    journal_sentence_set = set(journal_rows["sentence"].astype(str))
    assert journal_sentence_set.isdisjoint(set(val_data["sentence"].astype(str)))
    assert journal_sentence_set.isdisjoint(set(test_data["sentence"].astype(str)))

if DATASET_CHOICE in {"3", "4"}:
    synthetic_sentence_set = set(synthetic_rows["sentence"].astype(str))
    assert synthetic_sentence_set.isdisjoint(set(val_data["sentence"].astype(str)))
    assert synthetic_sentence_set.isdisjoint(set(test_data["sentence"].astype(str)))

# Επιβεβαίωση ότι μπήκαν στο train ακριβώς όσα external rows αναμένονται.
if DATASET_CHOICE == "2":
    assert (
        train_data["source"]
        .eq("Journal_Citation_Sentiment_Real")
        .sum()
        == len(journal_rows)
    )

if DATASET_CHOICE == "3":
    assert (
        train_data["source"]
        .eq("Synthetic_Citation_Sentiment")
        .sum()
        == len(synthetic_rows)
    )

if DATASET_CHOICE == "4":
    for source_name in [
        "Journal_Citation_Sentiment_Real",
        "Synthetic_Citation_Sentiment",
    ]:
        expected_count = int(
            combined_external_rows["source"].eq(source_name).sum()
        )
        actual_count = int(
            train_data["source"].eq(source_name).sum()
        )
        assert actual_count == expected_count

# ------------------------------------------------------------
# Split signatures.
# Across all 4 dataset modes (A0-A7), VAL/TEST hashes must be identical.
# ------------------------------------------------------------
def dataframe_signature(dataframe):
    payload = "\n".join(
        (
            dataframe["sentence"].astype(str)
            + "\t"
            + dataframe["polarity"].astype(str)
        ).tolist()
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


val_signature = dataframe_signature(val_data)
test_signature = dataframe_signature(test_data)

print("\n" + "=" * 70)
print("SPLIT CHECK")
print("=" * 70)

print("Experiment:", EXPERIMENT_TAG)
print("Loss strategy:", LOSS_LABEL)
print("Fixed Base + Clinical train rows πριν augmentation:", len(base_train_data))
print("Final train rows:", len(train_data))
print("Validation rows:", len(val_data))
print("Test rows:", len(test_data))

print("\nTraining polarity distribution:")
print(train_data["polarity"].value_counts().sort_index())

print("\nValidation polarity distribution:")
print(val_data["polarity"].value_counts().sort_index())

print("\nTest polarity distribution:")
print(test_data["polarity"].value_counts().sort_index())

print("\nClinical distribution in the fixed split:")
print(
    "Train / Validation / Test =",
    int(base_train_data["source"].eq(clinical_source_name).sum()),
    "/",
    int(val_data["source"].eq(clinical_source_name).sum()),
    "/",
    int(test_data["source"].eq(clinical_source_name).sum()),
)

print("\nValidation SHA-256:", val_signature)
print("Test SHA-256:", test_signature)

if DATASET_CHOICE in {"2", "4"}:
    print(
        "\nJournal real rows in train:",
        int(train_data["source"].eq("Journal_Citation_Sentiment_Real").sum())
    )
    print(
        "Journal real rows in validation:",
        int(val_data["source"].eq("Journal_Citation_Sentiment_Real").sum())
    )
    print(
        "Journal real rows in test:",
        int(test_data["source"].eq("Journal_Citation_Sentiment_Real").sum())
    )

if DATASET_CHOICE in {"3", "4"}:
    print(
        "\nSynthetic rows in train:",
        int(train_data["source"].eq("Synthetic_Citation_Sentiment").sum())
    )
    print(
        "Synthetic rows in validation:",
        int(val_data["source"].eq("Synthetic_Citation_Sentiment").sum())
    )
    print(
        "Synthetic rows in test:",
        int(test_data["source"].eq("Synthetic_Citation_Sentiment").sum())
    )


## 10. Standard CE / Weighted CE and validation metrics

In [ ]:
# ============================================================
# LOSS STRATEGY: STANDARD CE vs CLASS-WEIGHTED CE

# ============================================================
# Standard CE:
#   - No custom loss is passed to Trainer.
#   - Hugging Face uses the model's normal CrossEntropyLoss.
#
# Weighted CE:
#   - Class weights are computed ONLY from the FINAL train_data of
#     the selected experiment.
#   - sklearn's balanced rule is:
#       weight_c = N_train / (n_classes * N_c)
#   - Validation and test labels are NEVER used to compute weights.

# ============================================================
CLASS_IDS = np.array([0, 1, 2])

print("\nTraining-set polarity distribution:")
print(train_data["polarity"].value_counts().sort_index())

class_weights_np = None
class_weights = None

if USE_WEIGHTED_CE:
    class_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=CLASS_IDS,
        y=train_data["polarity"].to_numpy()
    )

    class_weights = torch.tensor(
        class_weights_np,
        dtype=torch.float32
    )

    print("\nLoss strategy: WEIGHTED CROSS ENTROPY")
    print("Class weights computed ONLY from the FINAL train_data:")
    for class_id, weight in zip(CLASS_IDS, class_weights_np):
        class_name = {0: "Negative", 1: "Neutral", 2: "Positive"}[int(class_id)]
        print(f"{class_name} ({class_id}): {weight:.6f}")

    def weighted_ce_loss(outputs, labels, num_items_in_batch=None):
        """
        Weighted CrossEntropy for Hugging Face Trainer.

        The weights are derived only from the final training set of the
        selected A0-A7 experiment and moved to the logits device at runtime.
        """
        logits = outputs.logits
        weights = class_weights.to(
            device=logits.device,
            dtype=logits.dtype
        )

        return torch.nn.functional.cross_entropy(
            logits,
            labels,
            weight=weights
        )
else:
    print("\nLoss strategy: STANDARD CROSS ENTROPY")
    print("No class weights are used. Trainer will use the model's default loss.")


def compute_metrics(eval_pred):
    """
    Metrics used during validation.

    Macro-F1 is the primary checkpoint-selection metric because
    the dataset is strongly imbalanced and minority-class
    performance is central to this experiment.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),
        "negative_f1": f1_score(
            (labels == 0).astype(int),
            (predictions == 0).astype(int),
            zero_division=0
        ),
        "neutral_f1": f1_score(
            (labels == 1).astype(int),
            (predictions == 1).astype(int),
            zero_division=0
        ),
        "positive_f1": f1_score(
            (labels == 2).astype(int),
            (predictions == 2).astype(int),
            zero_division=0
        ),
    }

#temp_data, test_data = train_test_split(df, test_size=0.3, stratify=df['polarity'], random_state=SEED)
#train_data, val_data = train_test_split(temp_data, test_size=0.2, stratify=temp_data['polarity'], random_state=SEED)


## 11. Dataset class, model selection and learning-rate configuration

In [ ]:
# Εμφάνιση των διαστάσεων των σετ δεδομένων.
print("Διαστάσεις σετ εκπαίδευσης:", train_data.shape)
print("Διαστάσεις σετ ελέγχου:", test_data.shape)
print("Διαστάσεις σετ επικύρωσης:", val_data.shape)

print("Το πρώτο δείγμα του train είναι: ", train_data.head(1))

# Κλάση για τη διαχείριση των δεδομένων
class PolarityDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Επιλογή μεταξύ 7 μοντέλων.
print("Επιλέξτε μοντέλο: 1 - BERT, 2 - DeBERTa, 3 - XLNet, 4 - RoBERTa, 5 - DistilBERT, 6 - ELECTRA, 7 - ALBERT, 8 - SciBERT")
ch = input("Επιλογή: ")

# Επιλογή αριθμού εποχών εκπαίδευσης.
EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

# Δεκτές εποχές απο 3 έως 10.
while not EPOCHS >= 3 or not EPOCHS <=10:
    EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης ")) # Αρχικά δίνω τιμή 10. Αν δεν υπάρχει βελτίωση στην εκπαίδευση, θα τερματιστεί νωρίτερα.

# Επιλογή αριθμού learning rate μόνο στο BASELINE A0.
if DATASET_CHOICE == "1" and LOSS_CHOICE == "1":
        learning_rate = input("Δώσε αριθμό για το learning rate hyperparameter: 1=1e-5, 2=2e-5, 3=3e-5, 4=5e-5 ")
        if learning_rate == "1":
            learning_rate = 1e-5
            tag_lr = "1e-5"
        elif learning_rate == "2":
            learning_rate = 2e-5
            tag_lr = "2e-5"
        elif learning_rate == "3":
            learning_rate = 3e-5
            tag_lr = "3e-5"
        elif learning_rate == "4":
            learning_rate = 5e-5
            tag_lr = "5e-5"
        else:
            learning_rate = 1e-5
            tag_lr = "1e-5"
else:
    tag_lr = "2e-5"             # Το καλύτερο learning rate που βγήκε στο experiment Α0.
    learning_rate = 2e-5        # Το καλύτερο learning rate που βγήκε στο experiment Α0.

print(f"Επιλέχθηκε learning rate = {learning_rate}")

if ch == "1":
    FOLDER_NAME = "BERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο BERT.
    MODEL_NAME = "bert-base-uncased"
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3) #hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3
    print("Χρησιμοποιείται το μοντέλο BERT.")
elif ch == "2":
    FOLDER_NAME = "DeBERTa" # Κατάλογος όπου αποθηκεύεται το μοντέλο DeBERTa.
    MODEL_NAME = "microsoft/deberta-v3-base"
    tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
    model = DebertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
    print("Χρησιμοποιείται το DeBERTa μοντέλο.")
elif ch == "3":
    FOLDER_NAME = "XLNet" # Κατάλογος όπου αποθηκεύεται το μοντέλο XLNet.
    MODEL_NAME = "xlnet-base-cased"
    tokenizer = XLNetTokenizer.from_pretrained(MODEL_NAME)
    model = XLNetForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το XLNet μοντέλο.")
elif ch == "4":
    FOLDER_NAME = "RoBERTa" # Κατάλογος όπου αποθηκεύεται το μοντέλο RoBERTa.
    MODEL_NAME = "roberta-base"
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
    model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3) #hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3
    print("Χρησιμοποιείται το RoBERTa μοντέλο.")
elif ch == "5":
    FOLDER_NAME = "DistilBERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο DistilBERT.
    MODEL_NAME = "distilbert-base-uncased"
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το DistilBERT μοντέλο.")
elif ch == "6":
    FOLDER_NAME = "ELECTRA" # Κατάλογος όπου αποθηκεύεται το μοντέλο ELECTRA.
    MODEL_NAME = "google/electra-base-discriminator"
    tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)
    model = ElectraForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το ELECTRA μοντέλο.")
elif ch == "7":
    FOLDER_NAME = "ALBERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο ALBERT.
    MODEL_NAME = "albert-base-v2"
    tokenizer = AlbertTokenizer.from_pretrained(MODEL_NAME)
    model = AlbertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το ALBERT μοντέλο.")
elif ch == "8":
    FOLDER_NAME = "SciBERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο SciBERT.
    MODEL_NAME = "allenai/scibert_scivocab_uncased"
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το SciBERT μοντέλο.")
else:
    raise ValueError("Μη έγκυρη επιλογή μοντέλου!")

# Διαφορετικό output name για κάθε experiment.
FOLDER_NAME = f"{FOLDER_NAME}_{EXPERIMENT_TAG}"
print("Experiment output tag:", FOLDER_NAME)


## 12. Special citation tokens, tokenization and training arguments

In [ ]:
# ============================================================
# SPECIAL TOKENS

# ============================================================

SPECIAL_CITATION_TOKENS = {
    "additional_special_tokens": [
        "[TARGET_CITATION]",
        "[OTHER_CITATION]",
    ]
}

num_added_tokens = tokenizer.add_special_tokens(
    SPECIAL_CITATION_TOKENS
)

model.resize_token_embeddings(len(tokenizer))

for special_token in [
    "[TARGET_CITATION]",
    "[OTHER_CITATION]",
]:
    token_ids = tokenizer.encode(
        special_token,
        add_special_tokens=False,
    )
    print(
        special_token,
        "->",
        tokenizer.tokenize(special_token),
        "-> IDs:",
        token_ids,
    )
    assert len(token_ids) == 1

# Προετοιμασία των δεδομένων.
train_encodings = tokenizer(train_data['sentence'].tolist(), truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_data['sentence'].tolist(), truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_data['sentence'].tolist(), truncation=True, padding=True, max_length=256)

# Δημιουργία του dataset.
train_dataset = PolarityDataset(train_encodings, train_data['polarity'].tolist())
test_dataset = PolarityDataset(test_encodings, test_data['polarity'].tolist())
val_dataset = PolarityDataset(val_encodings, val_data['polarity'].tolist())

# Εμφάνιση των tokens του train_encodings.
input_ids = train_encodings['input_ids'][0]   # το πρώτο δείγμα του train μετά τον τεμαχισμό.
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print(tokens)

from transformers import EarlyStoppingCallback

# Δυναμικός υπολογισμός warmup steps (π.χ. 10% των συνολικών steps)
total_steps = math.ceil(len(train_dataset) / 16) * EPOCHS
warmup_steps = int(total_steps * 0.1)

# Ορίσματα εκπαίδευσης.
training_args = TrainingArguments(
    output_dir = f'./results/{FOLDER_NAME}',  # ξεχωριστό output directory ανά experiment
    run_name=f"Sentiment_Analysis_{FOLDER_NAME}",
    num_train_epochs = EPOCHS,         # Αριθμός εποχών εκπαίδευσης.
    per_device_train_batch_size = 16,  # batch size για εκπαίδευση.
    per_device_eval_batch_size = 16,   # batch size για αξιολόγηση.
    warmup_steps = warmup_steps,                # Ορίζει τον αριθμό των βημάτων εκπαίδευσης για τα οποία ο ρυθμός μάθησης (learning rate) αυξάνεται σταδιακά από το 0 μέχρι να φτάσει στην προκαθορισμένη τιμή του.
    weight_decay = 0.1,              # Τεχνική που εφαρμόζεται στον αλγόριθμο βελτιστοποίησης για να αποτρέψει την υπερπροσαρμογή (overfitting) του μοντέλου.
    #logging_dir = './logs',            # directory for storing logs
    logging_steps = 50,
    #warmup_ratio = 0.1,
    eval_strategy = "epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=learning_rate,
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    optim="adamw_torch_fused"
)

#my_optimizer = Adam(model.parameters(), lr=5e-5)  # Προσαρμογή του learning rate.
#my_optimizer = bnb.optim.Adam32bit(model.parameters(), lr=1e-5)
#my_optimizer = bnb.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01) #ηταν 3e-5

from transformers import TrainerCallback

class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        # θα κρατήσουμε όλα τα logs με epoch, loss, eval_loss
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        epoch = logs.get("epoch", None)
        loss = logs.get("loss", None)

        # Validation metrics
        eval_loss = logs.get("eval_loss", None)
        eval_accuracy = logs.get("eval_accuracy", None)
        eval_macro_f1 = logs.get("eval_macro_f1", None)
        eval_negative_f1 = logs.get("eval_negative_f1", None)
        eval_neutral_f1 = logs.get("eval_neutral_f1", None)
        eval_positive_f1 = logs.get("eval_positive_f1", None)

        # κρατάμε μόνο logs που έχουν epoch ΚΑΙ τουλάχιστον ένα από loss / eval_loss
        if epoch is not None and (loss is not None or eval_loss is not None or eval_macro_f1 is not None):
            self.records.append({
                "epoch": epoch,
                "loss": loss,
                "eval_loss": eval_loss,
                "eval_accuracy": eval_accuracy,
                "eval_macro_f1": eval_macro_f1,
                "eval_negative_f1": eval_negative_f1,
                "eval_neutral_f1": eval_neutral_f1,
                "eval_positive_f1": eval_positive_f1
            })


## 13. Trainer construction and model training

In [ ]:
# Δημιουργία callback
loss_logger = LossLoggerCallback()

# Εκπαίδευση του μοντέλου με custom callback για logging του Loss.
# Δημιουργία κοινών Trainer arguments για ΟΛΑ τα A0-A7 experiments.
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "compute_metrics": compute_metrics,
    "callbacks": [loss_logger, EarlyStoppingCallback(early_stopping_patience=2)],
}

# ΜΟΝΟ στα weighted experiments (A1, A3, A5, A7)
# περνάμε custom weighted CrossEntropy loss.
# Στα standard experiments (A0, A2, A4, A6) δεν περνάμε
# compute_loss_func, οπότε χρησιμοποιείται το default CE του μοντέλου.
if USE_WEIGHTED_CE:
    trainer_kwargs["compute_loss_func"] = weighted_ce_loss

trainer = Trainer(**trainer_kwargs)

# Ορισμός του χρόνου έναρξης
start_time = time.time()

# Εκπαίδευση του μοντέλου
print("Ξεκινά η εκπαίδευση του μοντέλου...\n")
trainer.train()

# Ορισμός του χρόνου λήξης
end_time = time.time()

# Υπολογισμός του συνολικού χρόνου εκπαίδευσης
total_time = end_time - start_time

# Μετατροπή σε λεπτά & δευτερόλεπτα.
# Χρησιμοποιούμε // 60 για να βρούμε τα λεπτά και % 60 για τα υπόλοιπα δευτερόλεπτα.
# Το % είναι ο τελεστής υπολοίπου MOD.
# Το // κάνει ακέραια διαίρεση, επιστρέφοντας μόνο το ακέραιο μέρος.
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"\nΣυνολικός Χρόνος Εκπαίδευσης: {minutes} λεπτά και {seconds} δευτερόλεπτα ({training_args.num_train_epochs} εποχές)")


## 14. Training/validation loss logs

In [ ]:
# Τα raw logs από το callback
records = loss_logger.records

df_logs = pd.DataFrame(records)

# Διαχωρίζουμε train και val
train_df = df_logs[df_logs["loss"].notna()][["epoch", "loss"]]
train_df = train_df.rename(columns={"loss": "train_loss"})

#eval_df = df_logs[df_logs["eval_loss"].notna()][["epoch", "eval_loss"]]
#eval_df = eval_df.rename(columns={"eval_loss": "val_loss"})

eval_df = df_logs[
    df_logs["eval_loss"].notna()
][
    ["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1", "eval_negative_f1", "eval_neutral_f1", "eval_positive_f1"]
]

eval_df = eval_df.rename(columns={
    "eval_loss": "val_loss",
    "eval_accuracy": "val_accuracy",
    "eval_macro_f1": "val_macro_f1",
    "eval_negative_f1": "val_negative_f1",
    "eval_neutral_f1": "val_neutral_f1",
    "eval_positive_f1": "val_positive_f1"
})

# Κάνουμε merge ανά epoch
df_loss = pd.merge(train_df, eval_df, on="epoch", how="left")

# Τακτοποιούμε / στρογγυλοποιούμε
df_loss = df_loss.sort_values("epoch").reset_index(drop=True)
df_loss["train_loss"] = df_loss["train_loss"].astype(float)
df_loss["val_loss"] = df_loss["val_loss"].astype(float)
df_loss["val_accuracy"] = df_loss["val_accuracy"].astype(float)
df_loss["val_macro_f1"] = df_loss["val_macro_f1"].astype(float)
df_loss["val_negative_f1"] = df_loss["val_negative_f1"].astype(float)
df_loss["val_neutral_f1"] = df_loss["val_neutral_f1"].astype(float)
df_loss["val_positive_f1"] = df_loss["val_positive_f1"].astype(float)

print("To CSV είναι: ")
print(df_loss)

print("\nBest validation Macro-F1:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

# Αποθήκευση σε CSV για αυτό το μοντέλο
csv_path = f"/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/logs/loss_{FOLDER_NAME}_{tag_lr}.csv"
df_loss.to_csv(csv_path, index=False)
print("Saved loss CSV to:", csv_path)


## 15. Save model, test predictions, classification report and confusion matrix

In [ ]:
# Αποθήκευση του μοντέλου και του tokenizer (για πολικότητα).
model.save_pretrained(f'/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/pretrained_citation_polarity_recognition/{FOLDER_NAME}_{tag_lr}/saved_model')
tokenizer.save_pretrained(f'/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/pretrained_citation_polarity_recognition/{FOLDER_NAME}_{tag_lr}/saved_tokenizer')

print(f"\nΤο fine-tuned μοντέλο ({MODEL_NAME}) αποθηκεύτηκε επιτυχώς!")

# Προβλέψεις με το test set
predictions = trainer.predict(test_dataset)
predicted_classes = np.argmax(predictions.predictions, axis=-1)

# Classification Report
report = classification_report(test_data['polarity'], predicted_classes, target_names=['Negative', 'Neutral', 'Positive'])
print(report)

with open(f"/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/classification_report_txt_format/{FOLDER_NAME}_classification_report_{tag_lr}.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Heatmap
# Ενοποιημένο μέγεθος γραμματοσειράς
plt.rcParams.update({'font.size': 11})
plt.figure(figsize=(6, 4))

cm = confusion_matrix(test_data['polarity'], predicted_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()

#plt.title(f'Confusion Matrix - {FOLDER_NAME}')

# Αποθήκευση ως PDF υψηλής ανάλυσης.
plt.savefig(
    f'/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/fig/{FOLDER_NAME}_heatmap_CSA_{tag_lr}.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

# cmap = magma, cividis, Greys, Purples, Blues, Greens, Oranges, Reds

# Αποθήκευση των δεδομένων train_data['sentence'], test_data['sentence'] και val_data['sentence']
# σε αρχεία προκειμένου να επιβεβαιώσουμε ότι σε κάθε εκτέλεση του συγκεκριμένου σημειωματαρίου
# τα δεδομένα μας θα έιναι πάντα τα ίδια. Δηλαδή ίδιο train set, ίδιο test set και ίδιο validation set.
# Έτσι είναι πιο ακριβής η σύγκριση των μοντέλων.

# Αποθήκευση των δεδομένων σε αρχεία .txt


## 16. Training Loss vs Validation Loss plot

In [ ]:
# ============================================
# Training Loss vs Validation Loss
# ============================================

# Ρυθμίσεις εμφάνισης
sns.set(style="whitegrid", context="paper", font_scale=1.0)
plt.rcParams.update({'font.size': 11})

# Δημιουργία διαγράμματος
plt.figure(figsize=(6, 4))

plt.plot(
    df_loss["epoch"],
    df_loss["train_loss"],
    marker="o",
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    df_loss["epoch"],
    df_loss["val_loss"],
    marker="o",
    linewidth=2,
    label="Validation Loss"
)

# Ετικέτες αξόνων
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Σταθερός X-axis
plt.xticks(df_loss["epoch"])

# Σταθερός Y-axis
plt.ylim(0, 1.5)
plt.yticks(np.arange(0, 1.51, 0.1))

# Legend
plt.legend(fontsize=11)

# Grid
plt.grid(axis="both", linestyle="--", alpha=0.7)

plt.tight_layout()

# Αποθήκευση ως PDF υψηλής ανάλυσης
plt.savefig(
    f"/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/fig/"
    f"{FOLDER_NAME}_train_validation_loss_{tag_lr}.pdf",
    format="pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

"""train_data['sentence'].to_csv('train_sentences.txt', index=False, header=False)
test_data['sentence'].to_csv('test_sentences.txt', index=False, header=False)
val_data['sentence'].to_csv('val_sentences.txt', index=False, header=False)

print("Τα αρχεία αποθηκεύτηκαν επιτυχώς!")

from google.colab import files

files.download('train_sentences.txt')
files.download('test_sentences.txt')
files.download('val_sentences.txt')
"""
